In [6]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def scrape_market_overview():
    years = [2017, 2018, 2019, 2023, 2024, 2025]
    result_data = []
    
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.maximize_window()
    
    try:
        wait = WebDriverWait(driver, 15)

        for year in years:
            # nation_code: 전체는 빈 문자열, 국산은 'K'
            for category_name, nation_code in [("전체", ""), ("국산", "K")]:
                print(f"[진행중] {year}년 {category_name} 데이터 접속")
                
                # 사용자가 지정한 URL 패턴 적용 (sRepNationCd 변수 처리)
                url = f"https://www.kobis.or.kr/kobis/business/stat/boxs/findYearlyBoxOfficeList.do?loadEnd=0&searchType=search&sSearchYearFrom={year}&sMultiMovieYn=N&sRepNationCd={nation_code}"
                
                driver.get(url)
                
                try:
                    # 테이블 로딩 대기
                    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table.tbl_comm tbody tr")))
                    time.sleep(2)
                except Exception:
                    print(f"[오류] {year}년 {category_name} 페이지 로딩 실패")
                    continue

                rows = driver.find_elements(By.CSS_SELECTOR, "table.tbl_comm tbody tr")
                
                count = 0
                for row in rows:
                    if count >= 25: break
                    
                    cols = row.find_elements(By.TAG_NAME, "td")
                    if len(cols) < 8: continue

                    try:
                        try:
                            title = row.find_element(By.CSS_SELECTOR, "span.txt_ellip").text.strip()
                        except:
                            title = cols[1].text.strip()
                            
                        if not title: continue

                        revenue = cols[3].text.replace(",", "").strip()
                        audience = cols[5].text.replace(",", "").strip()
                        screens = cols[7].text.replace(",", "").strip()
                        
                        result_data.append({
                            "범주": category_name,
                            "연도": year,
                            "순위": count + 1,
                            "영화명": title,
                            "매출액": int(revenue) if revenue.isdigit() else 0,
                            "관객수": int(audience) if audience.isdigit() else 0,
                            "스크린수": int(screens) if screens.isdigit() else 0
                        })
                        count += 1
                        
                    except Exception:
                        continue
                        
        df = pd.DataFrame(result_data)
        df.to_csv("market_overview.csv", index=False, encoding="utf-8-sig")
        print("[완료] market_overview.csv 파일 생성됨")

    except Exception as e:
        print(f"[오류] 실행 중 문제 발생: {e}")
    finally:
        driver.quit()

if __name__ == "__main__":
    scrape_market_overview()

[진행중] 2017년 전체 데이터 접속
[진행중] 2017년 국산 데이터 접속
[진행중] 2018년 전체 데이터 접속
[진행중] 2018년 국산 데이터 접속
[진행중] 2019년 전체 데이터 접속
[진행중] 2019년 국산 데이터 접속
[진행중] 2023년 전체 데이터 접속
[진행중] 2023년 국산 데이터 접속
[진행중] 2024년 전체 데이터 접속
[진행중] 2024년 국산 데이터 접속
[진행중] 2025년 전체 데이터 접속
[진행중] 2025년 국산 데이터 접속
[완료] market_overview.csv 파일 생성됨


In [11]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
import chromedriver_autoinstaller

# 1. 브라우저 설정
print("[진행] 크롤링 준비 중...")
chromedriver_autoinstaller.install()
options = Options()
options.add_experimental_option("detach", True)
# options.add_argument('--headless') # 필요시 주석 해제
driver = webdriver.Chrome(options=options)
driver.maximize_window()

# 수집 설정
target_years = [2017, 2018, 2019, 2023, 2024, 2025]
target_genres = [
    "드라마", "액션", "사극", "코미디", "범죄", 
    "멜로/로맨스", "공포(호러)", "스릴러", "애니메이션", "미스터리"
]

all_data = []
missing_report = []

try:
    for year in target_years:
        print(f"\n[진행] {year}년 데이터 수집 시작...")
        
        # 쿼터 초기화 (장르별 전체 4개, 국산 3개)
        quotas = {g: {'total': 0, 'domestic': 0} for g in target_genres}
        
        # URL 접속 (sRepNationCd를 비워서 '전체' 영화 조회)
        url = f"https://www.kobis.or.kr/kobis/business/stat/boxs/findYearlyBoxOfficeList.do?loadEnd=0&searchType=search&sSearchYearFrom={year}&sMultiMovieYn=N&sRepNationCd="
        driver.get(url)
        
        # 테이블 로딩 대기
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "table.tbl_comm tbody tr"))
            )
            time.sleep(1)
        except:
            print(f"[경고] {year}년 페이지 로딩 실패")
            continue

        # 50개씩 보기 설정 (순위 낮은 영화도 조회하기 위함)
        try:
            cnt_select = Select(driver.find_element(By.NAME, "sPerPageFrom"))
            cnt_select.select_by_value("50")
            driver.execute_script("arguments[0].click();", driver.find_element(By.CLASS_NAME, "btn_blue"))
            time.sleep(2)
        except:
            pass

        checked_movies = 0
        page = 1
        max_check = 300 # 최대 300위까지 확인

        while checked_movies < max_check:
            # 행 가져오기
            try:
                WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.CSS_SELECTOR, "table.tbl_comm tbody tr")))
                rows = driver.find_elements(By.CSS_SELECTOR, "table.tbl_comm tbody tr")
            except:
                break

            for i in range(len(rows)):
                if checked_movies >= max_check: break
                
                # 모든 쿼터 달성 시 연도 루프 종료
                if all(q['total'] >= 4 and q['domestic'] >= 3 for q in quotas.values()):
                    break

                try:
                    # Stale Element 방지
                    rows = driver.find_elements(By.CSS_SELECTOR, "table.tbl_comm tbody tr")
                    if i >= len(rows): break
                    row = rows[i]
                    
                    cols = row.find_elements(By.TAG_NAME, "td")
                    if len(cols) < 7: continue

                    # 기본 정보
                    title = cols[1].text.strip()
                    if not title: continue 
                    
                    # 쉼표 제거 및 정수 변환
                    sales = int(cols[3].text.replace(",", "").strip())
                    audience = int(cols[5].text.replace(",", "").strip())
                    screens = int(cols[6].text.replace(",", "").strip())

                    # ==========================================================
                    # [사용자 제공 로직] 장르 수집: '요약정보' 텍스트를 파싱
                    # ==========================================================
                    found_genres = []
                    nation = "기타"
                    
                    try:
                        # 1. 영화 제목 클릭 (팝업 열기)
                        link = cols[1].find_element(By.TAG_NAME, "a")
                        driver.execute_script("arguments[0].click();", link)
                        
                        # 2. 팝업 대기
                        WebDriverWait(driver, 3).until(
                            EC.visibility_of_element_located((By.CLASS_NAME, "hd_layer"))
                        )
                        time.sleep(0.3) 

                        # 3. '요약정보' 텍스트 찾기
                        summary_elem = driver.find_element(By.XPATH, "//dt[contains(text(),'요약정보')]/following-sibling::dd")
                        summary_text = summary_elem.text # 예: "2019-01-23 | 15세관람가 | 코미디 | 한국 | 111분..."
                        
                        # 4. '|' 기준으로 자르고 내용 분석
                        info_parts = summary_text.split('|')
                        
                        # 장르 추출 (보통 인덱스 2번이지만, 안전하게 전체 텍스트에서 검색)
                        if len(info_parts) >= 3:
                            raw_genre = info_parts[2].strip() # "코미디, 액션"
                            # 콤마로 구분된 장르 분리
                            sub_genres = raw_genre.split(',')
                            for sub in sub_genres:
                                clean_sub = sub.strip()
                                if clean_sub in target_genres:
                                    found_genres.append(clean_sub)
                        
                        # 국적 추출 (요약정보 텍스트 내에서 검색)
                        if "한국" in summary_text:
                            nation = "한국"
                        elif any(x in summary_text for x in ["미국", "일본", "중국", "영국", "프랑스"]):
                            nation = "해외"
                        
                        # 5. 팝업 닫기
                        close_btn = driver.find_element(By.CSS_SELECTOR, "div.hd_layer a.close")
                        driver.execute_script("arguments[0].click();", close_btn)
                        time.sleep(0.2)
                        
                    except Exception as e:
                        # 실패 시 닫기 시도
                        try:
                            driver.execute_script("arguments[0].click();", driver.find_element(By.CSS_SELECTOR, "div.hd_layer a.close"))
                            time.sleep(0.2)
                        except:
                            pass
                        continue # 팝업 실패 시 다음 영화로
                    # ==========================================================

                    # 쿼터 체크 및 데이터 저장
                    is_korean = (nation == "한국")
                    
                    for g in set(found_genres):
                        # 1) 전체 쿼터 (4개)
                        if quotas[g]['total'] < 4:
                            all_data.append({
                                "범주": "전체", "영화명": title, "연도": year, "국적": nation,
                                "매출액": sales, "관객수": audience, "스크린수": screens, "장르": g
                            })
                            quotas[g]['total'] += 1
                            print(f"  + [전체] {g}: {title}")
                        
                        # 2) 국산 쿼터 (3개)
                        if is_korean and quotas[g]['domestic'] < 3:
                            all_data.append({
                                "범주": "국산", "영화명": title, "연도": year, "국적": nation,
                                "매출액": sales, "관객수": audience, "스크린수": screens, "장르": g
                            })
                            quotas[g]['domestic'] += 1
                            print(f"  + [국산] {g}: {title}")

                except Exception as e:
                    continue
                
                checked_movies += 1

            # 연도 종료 조건
            if all(q['total'] >= 4 and q['domestic'] >= 3 for q in quotas.values()):
                print(f" -> {year}년 모든 장르 쿼터 달성!")
                break
            
            # 다음 페이지 이동 (50개 다 봤는데 부족할 경우)
            try:
                page += 1
                next_page_xpath = f"//div[@class='paging']//a[contains(text(), '{page}')]"
                next_btn = driver.find_element(By.XPATH, next_page_xpath)
                driver.execute_script("arguments[0].click();", next_btn)
                time.sleep(2)
            except:
                print(f" -> {year}년 더 이상 페이지 없음.")
                break

        # 연도별 부족 현황 확인
        for g in target_genres:
            if quotas[g]['total'] < 4:
                missing_report.append(f"{year}년 [전체] {g}: {4-quotas[g]['total']}개 부족")
            if quotas[g]['domestic'] < 3:
                missing_report.append(f"{year}년 [국산] {g}: {3-quotas[g]['domestic']}개 부족")

except Exception as e:
    print(f"[오류 발생] {e}")

finally:
    driver.quit()

# 저장
if all_data:
    df = pd.DataFrame(all_data)
    df.to_csv("genre_analysis.csv", index=False, encoding="utf-8-sig")
    print("\n" + "="*50)
    print(f"[성공] genre_analysis.csv 저장 완료 (총 {len(df)}행)")
    
    if missing_report:
        print("[참고] 일부 부족한 데이터가 있습니다:")
        for msg in missing_report:
            print(f" - {msg}")
    else:
        print("[완벽] 모든 쿼터를 달성했습니다.")
    print("="*50)
else:
    print("[실패] 수집된 데이터가 없습니다.")

[진행] 크롤링 준비 중...

[진행] 2017년 데이터 수집 시작...
  + [전체] 드라마: 택시운전사
  + [국산] 드라마: 택시운전사
  + [전체] 드라마: 신과함께-죄와 벌
  + [국산] 드라마: 신과함께-죄와 벌
  + [전체] 액션: 공조
  + [국산] 액션: 공조
  + [전체] 액션: 스파이더맨: 홈 커밍
  + [전체] 범죄: 범죄도시
  + [국산] 범죄: 범죄도시
  + [전체] 액션: 범죄도시
  + [국산] 액션: 범죄도시
  + [전체] 액션: 군함도
  + [국산] 액션: 군함도
  + [전체] 드라마: 군함도
  + [국산] 드라마: 군함도
  + [전체] 범죄: 더 킹
  + [국산] 범죄: 더 킹
  + [전체] 드라마: 더 킹
  + [전체] 멜로/로맨스: 미녀와 야수
  + [전체] 코미디: 킹스맨: 골든 서클
  + [전체] 범죄: 꾼
  + [국산] 범죄: 꾼
  + [전체] 사극: 남한산성
  + [국산] 사극: 남한산성
  + [전체] 범죄: 분노의 질주: 더 익스트림
  + [전체] 스릴러: 분노의 질주: 더 익스트림
  + [전체] 멜로/로맨스: 너의 이름은.
  + [전체] 애니메이션: 너의 이름은.
  + [전체] 애니메이션: 슈퍼배드 3
  + [전체] 코미디: 아이 캔 스피크
  + [국산] 코미디: 아이 캔 스피크
  + [전체] 코미디: 캐리비안의 해적: 죽은 자는 말이 없다
  + [전체] 스릴러: 덩케르크
  + [전체] 스릴러: 살인자의 기억법
  + [국산] 스릴러: 살인자의 기억법
  + [전체] 코미디: 보안관
  + [국산] 코미디: 보안관
  + [전체] 애니메이션: 보스 베이비
  + [전체] 애니메이션: 모아나
  + [전체] 스릴러: 겟 아웃
  + [전체] 공포(호러): 겟 아웃
  + [전체] 미스터리: 겟 아웃
  + [전체] 공포(호러): 애나벨 : 인형의 주인
  + [국산] 코미디: 임금님의 사건수첩
  + [국산] 스릴러: 기억의 밤
  + [전체] 미스터리:

In [33]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import chromedriver_autoinstaller

# 1. 브라우저 설정
print("[진행] 온라인 데이터 크롤링 준비 중...")
chromedriver_autoinstaller.install()
options = Options()
options.add_experimental_option("detach", True)
driver = webdriver.Chrome(options=options)
driver.maximize_window()

# 수집 설정
years = [2017, 2018, 2019, 2023, 2024, 2025]

# 타겟 10개 장르 (HTML 텍스트와 정확히 일치)
target_genres = [
    "드라마", "액션", "사극", "코미디", "범죄", 
    "멜로/로맨스", "공포(호러)", "스릴러", "애니메이션", "미스터리"
]

result_data = []

try:
    # 1. 페이지 접속
    url = "https://www.vkobis.or.kr/statistics/selectGenreList.do"
    driver.get(url)
    wait = WebDriverWait(driver, 15)
    
    # 페이지 로딩 대기
    wait.until(EC.presence_of_element_located((By.ID, "searchStartDate")))
    time.sleep(1)

    for year in years:
        print(f"\n[진행] {year}년 온라인(VOD) 데이터 수집 시작...")
        
        try:
            # 2. 날짜 값 강제 주입 (JS)
            start_val = f"{year}-01"
            end_val = f"{year}-12"
            driver.execute_script(f"document.getElementById('searchStartDate').value = '{start_val}';")
            driver.execute_script(f"document.getElementById('searchEndDate').value = '{end_val}';")
            
            # 3. '전체' 버튼 리셋 (해제 -> 다시 선택)
            try:
                all_btn = driver.find_element(By.CSS_SELECTOR, "div.genre_chk a.all")
                # 1차 클릭 (혹시 켜져있으면 끄기, 꺼져있으면 켜기)
                driver.execute_script("arguments[0].click();", all_btn)
                time.sleep(0.5)
                # 2차 클릭 (확실하게 켜기)
                driver.execute_script("arguments[0].click();", all_btn)
                time.sleep(0.5)
            except Exception as e:
                print(f"  -> [경고] 버튼 조작 실패: {e}")

            # 4. 조회 버튼 클릭
            search_btn = driver.find_element(By.CSS_SELECTOR, "input.btnBora")
            driver.execute_script("arguments[0].click();", search_btn)
            
            # 5. 테이블 로딩 대기
            time.sleep(3) 
            # 제공해주신 HTML의 테이블 클래스 활용
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table.tbl tbody tr")))
            
            # 6. 테이블 데이터 파싱
            # vkobis 테이블은 보통 class="tbl" 또는 "tbl_bg_2n" 등을 가짐
            rows = driver.find_elements(By.CSS_SELECTOR, "table.tbl tbody tr")
            
            # 현재 연도의 데이터를 임시 저장할 딕셔너리
            current_year_data = {}
            
            for row in rows:
                cols = row.find_elements(By.TAG_NAME, "td")
                # 데이터 행은 컬럼이 5개임 (순위, 장르, 편수, 이용건수, 점유율)
                if len(cols) < 5: continue
                
                # 인덱스 1: 장르명 ( <a> 태그 안의 텍스트 )
                g_name = cols[1].text.strip()
                
                # 인덱스 3: 온라인 이용건수
                usage_raw = cols[3].text.replace(",", "").strip()
                
                # 인덱스 4: 점유율
                share_raw = cols[4].text.replace("%", "").strip()
                
                # 숫자 변환
                u_val = int(usage_raw) if usage_raw.isdigit() else 0
                try: s_val = float(share_raw)
                except: s_val = 0.0
                
                current_year_data[g_name] = {"이용건수": u_val, "점유율": s_val}
            
            # 7. 타겟 10개 장르만 추출하여 최종 리스트에 추가
            for target in target_genres:
                if target in current_year_data:
                    val = current_year_data[target]
                    result_data.append({
                        "연도": year,
                        "장르": target,
                        "온라인_이용건수": val["이용건수"],
                        "온라인_점유율": val["점유율"]
                    })
                    print(f"  + {target}: {val['이용건수']:,}건 ({val['점유율']}%)")
                else:
                    # 순위권에 없거나 데이터가 없는 경우 0 처리
                    result_data.append({
                        "연도": year,
                        "장르": target,
                        "온라인_이용건수": 0,
                        "온라인_점유율": 0.0
                    })
                    print(f"  - {target}: 데이터 없음 (0건)")

        except Exception as e:
            print(f"[경고] {year}년 처리 중 오류: {e}")
            continue

except Exception as e:
    print(f"[치명적 오류] {e}")

finally:
    driver.quit()

# 저장
if result_data:
    df = pd.DataFrame(result_data)
    filename = "online_stats.csv"
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print("\n" + "="*50)
    print(f"[완료] '{filename}' 저장되었습니다. (총 {len(df)}행)")
    print(df.head(10)) # 결과 미리보기
    print("="*50)
else:
    print("[실패] 수집된 데이터가 없습니다.")

[진행] 온라인 데이터 크롤링 준비 중...

[진행] 2017년 온라인(VOD) 데이터 수집 시작...
  + 드라마: 9,910,153건 (15.9%)
  + 액션: 15,925,170건 (25.6%)
  + 사극: 835,533건 (1.3%)
  + 코미디: 3,808,960건 (6.1%)
  + 범죄: 6,241,621건 (10.0%)
  + 멜로/로맨스: 4,281,614건 (6.9%)
  + 공포(호러): 1,674,240건 (2.7%)
  + 스릴러: 2,219,642건 (3.6%)
  + 애니메이션: 9,483,980건 (15.3%)
  + 미스터리: 1,410,266건 (2.3%)

[진행] 2018년 온라인(VOD) 데이터 수집 시작...
  + 드라마: 11,593,050건 (16.6%)
  + 액션: 18,411,786건 (26.4%)
  + 사극: 2,382,996건 (3.4%)
  + 코미디: 4,398,142건 (6.3%)
  + 범죄: 5,820,535건 (8.3%)
  + 멜로/로맨스: 4,619,572건 (6.6%)
  + 공포(호러): 2,405,381건 (3.4%)
  + 스릴러: 2,268,911건 (3.3%)
  + 애니메이션: 8,382,047건 (12.0%)
  + 미스터리: 1,956,359건 (2.8%)

[진행] 2019년 온라인(VOD) 데이터 수집 시작...
  + 드라마: 11,549,540건 (17.9%)
  + 액션: 14,942,162건 (23.2%)
  + 사극: 1,578,017건 (2.4%)
  + 코미디: 6,055,512건 (9.4%)
  + 범죄: 7,123,971건 (11.0%)
  + 멜로/로맨스: 3,894,537건 (6.0%)
  + 공포(호러): 2,193,730건 (3.4%)
  + 스릴러: 1,780,955건 (2.8%)
  + 애니메이션: 7,893,855건 (12.2%)
  + 미스터리: 1,367,189건 (2.1%)

[진행] 2023년 온라인(VOD) 데이터 수집 시작.

In [1]:
import os
import numpy as np
import pandas as pd

RAW_MARKET = "market_overview.csv"
RAW_GENRE  = "genre_analysis.csv"
RAW_ONLINE = "online_stats.csv"

OUT_DIR = "processed"
os.makedirs(OUT_DIR, exist_ok=True)

PRE_YEARS  = [2017, 2018, 2019]
POST_YEARS = [2023, 2024, 2025]
TARGET_YEARS = PRE_YEARS + POST_YEARS

def add_period_col(df, year_col="연도"):
    df = df.copy()
    df["기간"] = np.where(df[year_col].isin(PRE_YEARS), "Pre-COVID(2017-2019)",
                   np.where(df[year_col].isin(POST_YEARS), "Post-COVID(2023-2025)", "Other"))
    return df

def safe_div(a, b):
    a = a.astype(float)
    b = b.astype(float)
    return np.where(b == 0, np.nan, a / b)

def pct_change(pre, post):
    # (post-pre)/pre
    pre = pre.astype(float)
    post = post.astype(float)
    return np.where(pre == 0, np.nan, (post - pre) / pre)

# ---------------------------
# 1) Macro (market_overview)
# ---------------------------
market = pd.read_csv(RAW_MARKET)
market = market[market["연도"].isin(TARGET_YEARS)].copy()
market = add_period_col(market, "연도")

# 연도별 합계(Top25 합)
macro_yearly = (
    market.groupby(["범주", "연도"], as_index=False)[["매출액", "관객수", "스크린수"]]
          .sum()
)
macro_yearly["객단가(매출/관객)"] = safe_div(macro_yearly["매출액"], macro_yearly["관객수"])
macro_yearly["관객/상영(관객/스크린수)"] = safe_div(macro_yearly["관객수"], macro_yearly["스크린수"])
macro_yearly.to_csv(os.path.join(OUT_DIR, "macro_yearly.csv"), index=False, encoding="utf-8-sig")

# 기간별(Pre/Post) 평균: 연도별 합계를 만든 뒤 mean
macro_yearly2 = add_period_col(macro_yearly, "연도")
macro_period_avg = (
    macro_yearly2[macro_yearly2["기간"].isin(["Pre-COVID(2017-2019)", "Post-COVID(2023-2025)"])]
    .groupby(["범주", "기간"], as_index=False)
    .agg({
        "매출액": "mean",
        "관객수": "mean",
        "스크린수": "mean",
        "객단가(매출/관객)": "mean",
        "관객/상영(관객/스크린수)": "mean",
    })
)
macro_period_avg.to_csv(os.path.join(OUT_DIR, "macro_period_avg.csv"), index=False, encoding="utf-8-sig")

# 증감률 요약(카드/요약표용)
def make_summary(period_avg_df, id_cols=["범주"]):
    pre = period_avg_df[period_avg_df["기간"]=="Pre-COVID(2017-2019)"].set_index(id_cols)
    post = period_avg_df[period_avg_df["기간"]=="Post-COVID(2023-2025)"].set_index(id_cols)
    common = pre.index.intersection(post.index)

    pre = pre.loc[common]
    post = post.loc[common]

    out = pd.DataFrame(index=common).reset_index()
    metrics = [c for c in pre.columns if c not in ["기간"]]
    for m in metrics:
        out[f"{m}_Pre"] = pre[m].values
        out[f"{m}_Post"] = post[m].values
        out[f"{m}_증감률"] = pct_change(out[f"{m}_Pre"], out[f"{m}_Post"])
    return out

macro_summary = make_summary(macro_period_avg, ["범주"])
macro_summary.to_csv(os.path.join(OUT_DIR, "macro_summary.csv"), index=False, encoding="utf-8-sig")

# ---------------------------
# 2) Micro (genre_analysis)
# ---------------------------
genre = pd.read_csv(RAW_GENRE)
genre = genre[genre["연도"].isin(TARGET_YEARS)].copy()
genre = add_period_col(genre, "연도")

# 연도-장르 집계
genre_yearly = (
    genre.groupby(["범주", "연도", "장르"], as_index=False)
         .agg(매출액=("매출액", "sum"),
              관객수=("관객수", "sum"),
              스크린수=("스크린수", "sum"),
              영화수=("영화명", "count"))
)

# 파생지표(효율/단가)
genre_yearly["객단가(매출/관객)"] = safe_div(genre_yearly["매출액"], genre_yearly["관객수"])
genre_yearly["스크린당관객(관객/스크린)"] = safe_div(genre_yearly["관객수"], genre_yearly["스크린수"])
genre_yearly["스크린당매출(매출/스크린)"] = safe_div(genre_yearly["매출액"], genre_yearly["스크린수"])
genre_yearly["영화당관객(관객/영화수)"] = safe_div(genre_yearly["관객수"], genre_yearly["영화수"])
genre_yearly["영화당매출(매출/영화수)"] = safe_div(genre_yearly["매출액"], genre_yearly["영화수"])

# 점유율(연도/범주 내 장르 비중)
tot = genre_yearly.groupby(["범주","연도"], as_index=False)[["매출액","관객수"]].sum().rename(
    columns={"매출액":"총매출","관객수":"총관객"}
)
genre_yearly = genre_yearly.merge(tot, on=["범주","연도"], how="left")
genre_yearly["매출점유율"] = safe_div(genre_yearly["매출액"], genre_yearly["총매출"])
genre_yearly["관객점유율"] = safe_div(genre_yearly["관객수"], genre_yearly["총관객"])
genre_yearly.drop(columns=["총매출","총관객"], inplace=True)

genre_yearly.to_csv(os.path.join(OUT_DIR, "genre_yearly.csv"), index=False, encoding="utf-8-sig")

# 기간 평균(Pre/Post): 연도-장르 집계 후 mean
genre_yearly2 = add_period_col(genre_yearly, "연도")
genre_period_avg = (
    genre_yearly2[genre_yearly2["기간"].isin(["Pre-COVID(2017-2019)", "Post-COVID(2023-2025)"])]
    .groupby(["범주","기간","장르"], as_index=False)
    .agg({
        "매출액":"mean","관객수":"mean","스크린수":"mean","영화수":"mean",
        "객단가(매출/관객)":"mean",
        "스크린당관객(관객/스크린)":"mean",
        "스크린당매출(매출/스크린)":"mean",
        "영화당관객(관객/영화수)":"mean",
        "영화당매출(매출/영화수)":"mean",
        "매출점유율":"mean",
        "관객점유율":"mean",
    })
)
genre_period_avg.to_csv(os.path.join(OUT_DIR, "genre_period_avg.csv"), index=False, encoding="utf-8-sig")

# 장르별 증감률 요약
genre_summary = make_summary(genre_period_avg, ["범주","장르"])
genre_summary.to_csv(os.path.join(OUT_DIR, "genre_summary.csv"), index=False, encoding="utf-8-sig")

# ---------------------------
# 3) Online (online_stats)
# ---------------------------
online = pd.read_csv(RAW_ONLINE)
online = online[online["연도"].isin(TARGET_YEARS)].copy()
online = add_period_col(online, "연도")

online_yearly = (
    online.groupby(["연도","장르"], as_index=False)
          .agg(온라인_이용건수=("온라인_이용건수","sum"),
               온라인_점유율=("온라인_점유율","mean"))
)
online_yearly = add_period_col(online_yearly, "연도")
online_yearly.to_csv(os.path.join(OUT_DIR, "online_yearly.csv"), index=False, encoding="utf-8-sig")

online_period_avg = (
    online_yearly[online_yearly["기간"].isin(["Pre-COVID(2017-2019)","Post-COVID(2023-2025)"])]
    .groupby(["기간","장르"], as_index=False)
    .agg({"온라인_이용건수":"mean","온라인_점유율":"mean"})
)
online_period_avg.to_csv(os.path.join(OUT_DIR, "online_period_avg.csv"), index=False, encoding="utf-8-sig")

online_summary = make_summary(online_period_avg, ["장르"])
online_summary.to_csv(os.path.join(OUT_DIR, "online_summary.csv"), index=False, encoding="utf-8-sig")

# ---------------------------
# 4) Strategy Recommendation Table (All / Korean)
#   - 오프라인 장르 충격 + 온라인 장르 변화 결합
# ---------------------------
def build_strategy_table(scope_label):
    # scope_label: "전체" or "국산"
    gsum = genre_summary[genre_summary["범주"]==scope_label].copy()
    osum = online_summary.copy()

    # 매칭
    merged = gsum.merge(osum, on="장르", how="left", suffixes=("","_online"))
    # 핵심 지표 선택(관객/매출 증감률 + 온라인이용/점유율 증감률)
    merged["오프라인_관객증감률"] = merged["관객수_증감률"]
    merged["오프라인_매출증감률"] = merged["매출액_증감률"]
    merged["온라인_이용증감률"] = merged.get("온라인_이용건수_증감률", np.nan)
    merged["온라인_점유율증감률"] = merged.get("온라인_점유율_증감률", np.nan)

    # 추천 로직(규칙 기반, 설명 가능하게)
    # 기준: 오프라인 관객 급감(<= -0.30) + 온라인 이용 증가(>= 0) 또는 점유율 증가(>0)면 OTT 우선
    # 오프라인 안정(>= -0.10)이면 극장 우선
    # 그 외 혼합/재검토
    def reco_row(r):
        off = r["오프라인_관객증감률"]
        on_u = r["온라인_이용증감률"]
        on_s = r["온라인_점유율증감률"]
        if pd.notna(off) and off <= -0.30 and ((pd.notna(on_u) and on_u >= 0) or (pd.notna(on_s) and on_s > 0)):
            return "OTT 우선"
        if pd.notna(off) and off >= -0.10:
            return "극장 우선"
        return "혼합/재검토"

    merged["추천"] = merged.apply(reco_row, axis=1)

    # 근거 한 줄(앱에서 바로 보여주기 좋게)
    def fmt_pct(x):
        if pd.isna(x): return "NA"
        return f"{x*100:+.1f}%"

    merged["근거"] = (
        "오프라인 관객 " + merged["오프라인_관객증감률"].apply(fmt_pct) +
        ", 온라인 이용 " + merged["온라인_이용증감률"].apply(fmt_pct) +
        ", 온라인 점유율 " + merged["온라인_점유율증감률"].apply(fmt_pct)
    )

    # 보기 좋게 정렬(오프라인 충격 큰 순)
    merged = merged.sort_values(by="오프라인_관객증감률").reset_index(drop=True)

    keep_cols = [
        "장르","추천","근거",
        "오프라인_관객증감률","오프라인_매출증감률",
        "온라인_이용증감률","온라인_점유율증감률",
        "관객수_Pre","관객수_Post","매출액_Pre","매출액_Post"
    ]
    for c in keep_cols:
        if c not in merged.columns:
            merged[c] = np.nan
    return merged[keep_cols]

strategy_all = build_strategy_table("전체")
strategy_kor = build_strategy_table("국산")

strategy_all.to_csv(os.path.join(OUT_DIR, "strategy_reco_all.csv"), index=False, encoding="utf-8-sig")
strategy_kor.to_csv(os.path.join(OUT_DIR, "strategy_reco_korean.csv"), index=False, encoding="utf-8-sig")

print("✅ Done. Created files in:", OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))


✅ Done. Created files in: processed
['genre_period_avg.csv', 'genre_summary.csv', 'genre_yearly.csv', 'macro_period_avg.csv', 'macro_summary.csv', 'macro_yearly.csv', 'online_period_avg.csv', 'online_summary.csv', 'online_yearly.csv', 'strategy_reco_all.csv', 'strategy_reco_korean.csv']


In [ ]:
%%writefile movie.py
import os
import re
import numpy as np
import pandas as pd
import streamlit as st
import plotly.express as px

st.set_page_config(
    page_title="코로나 이후 영화 산업의 장르별 타격 영향 분석",
    page_icon="🎬",
    layout="wide"
)

PRE_YEARS = [2017, 2018, 2019]
POST_YEARS = [2023, 2024, 2025]

FILE_MARKET = "market_overview.csv"
FILE_GENRE  = "genre_analysis.csv"
FILE_ONLINE = "online_stats.csv"

# =========================
# Utils
# =========================
def _clean_col(c: str) -> str:
    return re.sub(r"\s+", "", str(c)).strip()

def _to_number(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace(",", "").replace(" ", "")
    s = re.sub(r"[^\d\.\-]", "", s)
    if s in ("", "-", ".", "-."):
        return np.nan
    try:
        return float(s)
    except:
        return np.nan

def _find_col(df, patterns):
    cols = list(df.columns)
    cleaned = {c: _clean_col(c) for c in cols}
    for p in patterns:
        try:
            rgx = re.compile(p)
            for c in cols:
                if rgx.search(cleaned[c]):
                    return c
        except:
            for c in cols:
                if p in cleaned[c]:
                    return c
    return None

def _pct_change(pre, post):
    if pd.isna(pre) or pre == 0 or pd.isna(post):
        return np.nan
    return (post - pre) / pre

def _fmt_pct(x):
    if pd.isna(x): return "NA"
    return f"{x*100:+.1f}%"

def _fmt_int(x):
    if pd.isna(x): return "NA"
    try:
        return f"{int(round(float(x))):,}"
    except:
        return str(x)

def _fmt_money(x):
    if pd.isna(x): return "NA"
    try:
        return f"{float(x):,.0f}"
    except:
        return str(x)

def _period_label(y: int) -> str:
    y = int(y)
    if y in PRE_YEARS:
        return "코로나 전"
    if y in POST_YEARS:
        return "코로나 후"
    return "기타"

def _safe_pct_text(v):
    return "NA" if pd.isna(v) else f"{v*100:+.1f}%"

def _safe_pp_text(v):
    return "NA" if pd.isna(v) else f"{v*100:+.2f}%p"

# =========================
# Charts
# =========================
def _thin_barh(df, x, y, title, x_is_pct=True, height=720):
    d = df.copy()
    if x_is_pct:
        d["_text"] = d[x].apply(lambda v: "" if pd.isna(v) else f"{v*100:+.1f}%")
    else:
        d["_text"] = d[x].apply(lambda v: "" if pd.isna(v) else f"{v:,.0f}")

    fig = px.bar(
        d, x=x, y=y,
        orientation="h",
        title=title,
        template="plotly_white",
        text="_text",
        color=x,
        color_continuous_scale="RdYlGn"
    )
    fig.update_layout(
        height=height,
        bargap=0.78,
        margin=dict(l=10, r=90, t=80, b=10),
        coloraxis_showscale=False,
        uniformtext_minsize=12,
        uniformtext_mode="show",
        title_font_size=20
    )
    fig.update_traces(textposition="outside", cliponaxis=False, textfont_size=14)
    fig.update_xaxes(tickformat=".0%" if x_is_pct else ",", title=None)
    fig.update_yaxes(title=None)
    st.plotly_chart(fig, use_container_width=True)

def _line(df, x, y, color, title, height=460):
    fig = px.line(
        df, x=x, y=y,
        color=color,
        markers=True,
        template="plotly_white",
        title=title
    )
    fig.update_layout(
        height=height,
        margin=dict(l=10, r=10, t=80, b=10),
        title_font_size=20
    )
    fig.update_xaxes(tickmode="linear", title=None)
    fig.update_yaxes(tickformat=",", title=None)
    st.plotly_chart(fig, use_container_width=True)

def _group_bar_pct(df, x, y, color, title, height=520):
    d = df.copy()
    d["_text"] = d[y].apply(lambda v: "" if pd.isna(v) else f"{v*100:+.1f}%")
    fig = px.bar(
        d, x=x, y=y,
        color=color,
        barmode="group",
        title=title,
        template="plotly_white",
        text="_text"
    )
    fig.update_layout(
        height=height,
        bargap=0.60,
        bargroupgap=0.55,
        margin=dict(l=10, r=10, t=80, b=10),
        uniformtext_minsize=12,
        uniformtext_mode="show",
        title_font_size=20
    )
    fig.update_traces(textposition="outside", cliponaxis=False, textfont_size=14)
    fig.update_yaxes(tickformat=".0%", title=None)
    fig.update_xaxes(title=None)
    st.plotly_chart(fig, use_container_width=True)

def _scatter_with_lines(df, x, y, text, title, vline=None, hline=None, height=650):
    fig = px.scatter(
        df, x=x, y=y,
        text=text,
        hover_name=text,
        template="plotly_white",
        title=title
    )
    fig.update_traces(textposition="top center", textfont_size=13)
    fig.update_layout(height=height, margin=dict(l=10, r=10, t=80, b=10), title_font_size=20)
    fig.update_xaxes(
    tickformat=".0%",
    title="오프라인 관객 변화율")
    fig.update_yaxes(
    tickformat=".0%",
    title="온라인 점유율 변화")


    if vline is not None:
        fig.add_vline(x=vline, line_width=2, line_dash="dash")
    if hline is not None:
        fig.add_hline(y=hline, line_width=2, line_dash="dash")

    st.plotly_chart(fig, use_container_width=True)

# =========================
# Loaders
# =========================
@st.cache_data(show_spinner=False)
def load_market():
    if not os.path.exists(FILE_MARKET):
        return None, f"{FILE_MARKET} 파일을 찾을 수 없습니다."
    df = pd.read_csv(FILE_MARKET)

    year_col  = _find_col(df, [r"연도", r"year"])
    scope_col = _find_col(df, [r"범주", r"구분", r"scope", r"type", r"category"])
    sales_col = _find_col(df, [r"매출", r"sales", r"revenue"])
    audi_col  = _find_col(df, [r"관객", r"aud", r"audience"])
    scrn_col  = _find_col(df, [r"스크린", r"상영", r"screen", r"show"])

    if year_col is None or scope_col is None or sales_col is None or audi_col is None:
        return None, f"{FILE_MARKET} 컬럼 인식 실패"

    out = df.rename(columns={
        year_col: "연도",
        scope_col: "범주",
        sales_col: "매출액",
        audi_col: "관객수",
        scrn_col: "스크린수" if scrn_col else scrn_col
    }).copy()

    out["연도"] = out["연도"].apply(_to_number).astype("Int64")
    out["매출액"] = out["매출액"].apply(_to_number)
    out["관객수"] = out["관객수"].apply(_to_number)
    out["스크린수"] = out["스크린수"].apply(_to_number) if scrn_col else np.nan

    out["범주"] = out["범주"].astype(str).str.replace(" ", "")
    out["범주"] = out["범주"].replace({
        "전체(국산+해외)": "전체",
        "전체(국산+외화)": "전체",
        "국산영화": "국산",
        "한국영화": "국산",
        "국내": "국산"
    })

    out = out[out["연도"].isin(PRE_YEARS + POST_YEARS)].copy()
    return out, None

@st.cache_data(show_spinner=False)
def load_genre():
    if not os.path.exists(FILE_GENRE):
        return None, f"{FILE_GENRE} 파일을 찾을 수 없습니다."
    df = pd.read_csv(FILE_GENRE)

    year_col  = _find_col(df, [r"연도", r"year"])
    genre_col = _find_col(df, [r"장르", r"genre"])
    scope_col = _find_col(df, [r"범주", r"구분", r"type", r"category"])
    sales_col = _find_col(df, [r"매출", r"sales", r"revenue"])
    audi_col  = _find_col(df, [r"관객", r"aud", r"audience"])
    scrn_col  = _find_col(df, [r"스크린", r"상영", r"screen", r"show"])

    if year_col is None or genre_col is None or sales_col is None or audi_col is None:
        return None, f"{FILE_GENRE} 컬럼 인식 실패"

    out = df.rename(columns={
        year_col: "연도",
        genre_col: "장르",
        scope_col: "범주" if scope_col else scope_col,
        sales_col: "매출액",
        audi_col: "관객수",
        scrn_col: "스크린수" if scrn_col else scrn_col
    }).copy()

    out["연도"] = out["연도"].apply(_to_number).astype("Int64")
    out["장르"] = out["장르"].astype(str).str.strip()
    out["매출액"] = out["매출액"].apply(_to_number)
    out["관객수"] = out["관객수"].apply(_to_number)
    out["스크린수"] = out["스크린수"].apply(_to_number) if scrn_col else np.nan

    if scope_col:
        out["범주"] = out["범주"].astype(str).str.replace(" ", "")
        out["범주"] = out["범주"].replace({
            "전체(국산+해외)": "전체",
            "전체(국산+외화)": "전체",
            "국산영화": "국산",
            "한국영화": "국산",
            "국내": "국산"
        })
    else:
        out["범주"] = "전체"

    out = out[out["연도"].isin(PRE_YEARS + POST_YEARS)].copy()
    return out, None

@st.cache_data(show_spinner=False)
def load_online():
    if not os.path.exists(FILE_ONLINE):
        return None, f"{FILE_ONLINE} 파일을 찾을 수 없습니다."
    df = pd.read_csv(FILE_ONLINE)

    year_col  = _find_col(df, [r"연도", r"year"])
    genre_col = _find_col(df, [r"장르", r"genre"])
    share_col = _find_col(df, [r"점유율", r"share", r"비중"])

    if year_col is None or genre_col is None or share_col is None:
        return None, f"{FILE_ONLINE} 컬럼 인식 실패"

    out = df.rename(columns={year_col:"연도", genre_col:"장르", share_col:"점유율"}).copy()
    out["연도"] = out["연도"].apply(_to_number).astype("Int64")
    out["장르"] = out["장르"].astype(str).str.strip()
    out["점유율"] = out["점유율"].apply(_to_number)

    if out["점유율"].dropna().max() > 1.5:
        out["점유율"] = out["점유율"] / 100.0

    out = out[out["연도"].isin(PRE_YEARS + POST_YEARS)].copy()
    return out, None

# =========================
# Indicators
# =========================
@st.cache_data(show_spinner=False)
def build_market_indicators(df_market):
    year_sum = df_market.groupby(["연도","범주"], as_index=False)[["매출액","관객수","스크린수"]].sum()
    year_sum["기간"] = year_sum["연도"].apply(_period_label)

    period_avg = year_sum.groupby(["기간","범주"], as_index=False)[["매출액","관객수","스크린수"]].mean(numeric_only=True)

    rows = []
    for scope in sorted(period_avg["범주"].unique()):
        pre = period_avg[(period_avg["범주"]==scope) & (period_avg["기간"]=="코로나 전")]
        post= period_avg[(period_avg["범주"]==scope) & (period_avg["기간"]=="코로나 후")]
        if pre.empty or post.empty:
            continue
        pre = pre.iloc[0]; post = post.iloc[0]
        rows.append({
            "범주": scope,
            "매출액_전": pre["매출액"], "매출액_후": post["매출액"], "매출액_증감률": _pct_change(pre["매출액"], post["매출액"]),
            "관객수_전": pre["관객수"], "관객수_후": post["관객수"], "관객수_증감률": _pct_change(pre["관객수"], post["관객수"]),
            "스크린수_전": pre["스크린수"], "스크린수_후": post["스크린수"], "스크린수_증감률": _pct_change(pre["스크린수"], post["스크린수"]),
        })

    return year_sum, period_avg, pd.DataFrame(rows)

@st.cache_data(show_spinner=False)
def build_genre_indicators(df_genre):
    df = df_genre.copy()
    df["기간"] = df["연도"].apply(_period_label)
    agg = df.groupby(["기간","범주","장르"], as_index=False)[["매출액","관객수","스크린수"]].mean(numeric_only=True)

    rows = []
    for scope in sorted(agg["범주"].unique()):
        for g in sorted(agg["장르"].unique()):
            pre = agg[(agg["범주"]==scope) & (agg["장르"]==g) & (agg["기간"]=="코로나 전")]
            post= agg[(agg["범주"]==scope) & (agg["장르"]==g) & (agg["기간"]=="코로나 후")]
            if pre.empty or post.empty:
                continue
            pre = pre.iloc[0]; post = post.iloc[0]
            rows.append({
                "범주": scope,
                "장르": g,
                "관객수_증감률": _pct_change(pre["관객수"], post["관객수"]),
                "매출액_증감률": _pct_change(pre["매출액"], post["매출액"]),
                "스크린수_증감률": _pct_change(pre["스크린수"], post["스크린수"]),
            })
    return agg, pd.DataFrame(rows)

@st.cache_data(show_spinner=False)
def build_online_indicators(df_online):
    df = df_online.copy()
    df["기간"] = df["연도"].apply(_period_label)
    period_avg = df.groupby(["기간","장르"], as_index=False)["점유율"].mean()

    rows = []
    for g in sorted(period_avg["장르"].unique()):
        pre = period_avg[(period_avg["장르"]==g) & (period_avg["기간"]=="코로나 전")]
        post= period_avg[(period_avg["장르"]==g) & (period_avg["기간"]=="코로나 후")]
        if pre.empty or post.empty:
            continue
        pre_v = float(pre.iloc[0]["점유율"])
        post_v = float(post.iloc[0]["점유율"])
        rows.append({
            "장르": g,
            "점유율_전": pre_v,
            "점유율_후": post_v,
            "점유율_변화": post_v - pre_v,
        })

    return period_avg, pd.DataFrame(rows)

def _recommend_rule(offline_change, online_delta, x_thr, y_thr):
    if pd.isna(offline_change):
        return "재검토"
    if (offline_change <= x_thr) and (not pd.isna(online_delta)) and (online_delta >= y_thr):
        return "OTT"
    if offline_change > x_thr:
        return "극장"
    return "추가 검토"

# =========================
# Auto memos (그래프 결과 요약)
# =========================
def market_memo(m_change: pd.DataFrame) -> str:
    if m_change is None or m_change.empty:
        return "요약 생성 불가"
    d = m_change.set_index("범주")
    if ("전체" not in d.index) or ("국산" not in d.index):
        # 한 종류만 있을 때
        k = d.index[0]
        s = d.loc[k]
        return f"{k} 매출 변화율 {_safe_pct_text(s['매출액_증감률'])}, 관객 변화율 {_safe_pct_text(s['관객수_증감률'])}"
    all_s = d.loc["전체"]
    kor_s = d.loc["국산"]

    sales_gap = kor_s["매출액_증감률"] - all_s["매출액_증감률"]
    aud_gap   = kor_s["관객수_증감률"] - all_s["관객수_증감률"]

    sales_rel = "국산 감소폭이 더 큼" if (not pd.isna(sales_gap) and sales_gap < 0) else "국산 감소폭이 더 작음"
    aud_rel   = "국산 감소폭이 더 큼" if (not pd.isna(aud_gap) and aud_gap < 0) else "국산 감소폭이 더 작음"

    return (
        f"매출 변화율 전체 {_safe_pct_text(all_s['매출액_증감률'])}, 국산 {_safe_pct_text(kor_s['매출액_증감률'])}, {sales_rel}. "
        f"관객 변화율 전체 {_safe_pct_text(all_s['관객수_증감률'])}, 국산 {_safe_pct_text(kor_s['관객수_증감률'])}, {aud_rel}."
    )

def genre_memo(tmp: pd.DataFrame) -> str:
    if tmp is None or tmp.empty:
        return "요약 생성 불가"

    d = tmp.dropna(subset=["관객수_증감률"]).copy()
    if len(d) < 4:
        return "장르별 비교를 위한 데이터가 충분하지 않다."

    # 감소폭 큰 장르 2개
    worst = d.nsmallest(2, "관객수_증감률")[["장르","관객수_증감률"]].values.tolist()
    # 감소폭 작은 장르 2개
    best  = d.nlargest(2, "관객수_증감률")[["장르","관객수_증감률"]].values.tolist()

    w1, w2 = worst
    b1, b2 = best

    return (
        f"{w1[0]}와 {w2[0]} 장르는 관객 변화율이 각각 "
        f"{_safe_pct_text(w1[1])}, {_safe_pct_text(w2[1])}로 나타나 "
        f"분석 대상 장르 중 감소폭이 가장 큰 것으로 확인된다. "
        f"반면 {b1[0]}와 {b2[0]} 장르는 "
        f"{_safe_pct_text(b1[1])}, {_safe_pct_text(b2[1])} 수준으로 "
        f"상대적으로 관객 감소가 제한적인 장르로 분류된다."
    )


def online_memo(o_change: pd.DataFrame) -> str:
    if o_change is None or o_change.empty:
        return "요약 생성 불가"
    d = o_change.dropna(subset=["점유율_변화"]).copy()
    if d.empty:
        return "요약 생성 불가"
    down = d.nsmallest(2, "점유율_변화")[["장르","점유율_변화"]].values.tolist()
    up   = d.nlargest(2, "점유율_변화")[["장르","점유율_변화"]].values.tolist()
    dn_txt = ", ".join([f"{g} {_safe_pp_text(v)}" for g, v in down])
    up_txt = ", ".join([f"{g} {_safe_pp_text(v)}" for g, v in up])
    return f"점유율 하락 상위 {dn_txt}. 점유율 상승 상위 {up_txt}."

def strategy_memo(summary_df: pd.DataFrame) -> str:
    if summary_df is None or summary_df.empty or "제언" not in summary_df.columns:
        return "요약 생성 불가"
    cnt = summary_df["제언"].value_counts().to_dict()
    ott = cnt.get("OTT", 0)
    th  = cnt.get("극장", 0)
    ex  = cnt.get("추가 검토", 0)
    re  = cnt.get("재검토", 0)
    return f"제언 분류 결과 OTT {ott}개, 극장 {th}개, 추가 검토 {ex}개, 재검토 {re}개."

# =========================
# Load data
# =========================
df_market, err = load_market()
if err: st.error(err); st.stop()

df_genre, err = load_genre()
if err: st.error(err); st.stop()

df_online, err = load_online()
if err: st.error(err); st.stop()

m_year, m_period, m_change = build_market_indicators(df_market)
g_agg, g_change = build_genre_indicators(df_genre)
o_period, o_change = build_online_indicators(df_online)

# =========================
# Summary table
# =========================
base_scope = "국산" if "국산" in g_change["범주"].unique() else "전체"
base_label = "국산 영화 기준" if base_scope == "국산" else "전체 기준"

summary_df = pd.DataFrame()
x_thr = None
y_thr = None

if not o_change.empty:
    off = g_change[g_change["범주"]==base_scope][["장르","관객수_증감률","매출액_증감률"]].copy()
    merged = off.merge(o_change[["장르","점유율_변화"]], on="장르", how="left").dropna()
    if not merged.empty:
        x_thr = float(merged["관객수_증감률"].quantile(0.33))
        y_thr = float(merged["점유율_변화"].median())
        merged["제언"] = merged.apply(lambda r: _recommend_rule(r["관객수_증감률"], r["점유율_변화"], x_thr, y_thr), axis=1)
        summary_df = merged.rename(columns={
            "관객수_증감률": "오프라인 관객 변화율",
            "매출액_증감률": "오프라인 매출 변화율",
            "점유율_변화": "온라인 점유율 변화"
        }).copy()

# =========================
# Navigation
# =========================
st.sidebar.title("목차")
page = st.sidebar.radio(
    "파트 선택",
    ["연구배경 및 필요성", "프로젝트 진행과정", "자료 해석", "프로젝트 성과", "프로젝트 기대효과"],
    index=0
)

st.title("코로나 이후 영화 산업의 장르별 타격 영향 분석")
st.caption("코로나 전 2017–2019 | 코로나 후 2023–2025")

# =========================
# Pages
# =========================
if page == "연구배경 및 필요성":
    st.markdown("## 연구배경 및 필요성")
    st.markdown("""
코로나19 이후 영화 티켓 가격이 상승하면서 관람 비용이 증가하였다. 관람 비용 증가는 수요 감소로 이어질 수 있으며, 이는 관객수 감소로 확인된다. 매출액은 티켓 단가 상승 효과가 포함되므로, 시장 변화는 매출액과 관객수를 함께 비교하여 판단한다. 본 프로젝트는 코로나 전과 코로나 후를 비교하여 영화산업의 규모 변화와 수요 변화가 동시에 발생했는지 확인한다.  

또한 전체 시장과 국산 영화 시장의 충격은 동일하지 않을 수 있다. 전체 시장은 해외 흥행작 성과로 일부 완충될 수 있으나, 국산 영화는 관객 이탈의 영향을 더 크게 받을 수 있다. 따라서 동일한 비교 구간에서 국산의 감소폭이 더 큰지 여부를 별도로 확인한다.  

시장 쇠퇴가 확인되더라도 장르별 충격은 다르게 나타난다. 본 프로젝트는 수집한 10개 장르를 기준으로 장르별 타격 정도를 변화율 중심으로 비교하고, 결과를 바탕으로 장르별 개봉 전략을 제안한다.
""")
    st.info("메모: 분석 흐름은 시장 변화 확인 후 장르별 타격을 중심으로 정리한다.")

elif page == "프로젝트 진행과정":
    st.markdown("## 프로젝트 진행과정")

    st.markdown("### 데이터 수집")
    st.markdown("""
- KOBIS 영화관입장권통합전산망에서 연도별 박스오피스 데이터를 수집하였다.  
- 전체와 국산을 구분하여 2017–2019, 2023–2025 연도별 상위 영화 데이터를 확보하였다.  
- 장르별 분석을 위해 주요 장르를 기준으로 별도 조회하여 장르별 흥행 데이터를 확보하였다.  
- VKOBIS 온라인상영관통합전산망에서 장르별 온라인 이용 점유율 데이터를 수집하였다.
""")

    st.markdown("### 수집 방식")
    st.markdown("""
- Selenium 기반 자동화를 통해 페이지 로딩 대기, 테이블 요소 탐색, 상위 N개 행 추출, CSV 저장 과정을 수행하였다.
""")

    st.info("메모: 아래 코드는 market_overview.csv 생성에 사용한 수집 코드 일부 발췌이다.")

    excerpt = r'''
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

def scrape_market_overview():
    years = [2017, 2018, 2019, 2023, 2024, 2025]
    result_data = []
    
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.maximize_window()
    
    try:
        wait = WebDriverWait(driver, 15)

        for year in years:
            for category_name, nation_code in [("전체", ""), ("국산", "K")]:
                url = f"https://www.kobis.or.kr/kobis/business/stat/boxs/findYearlyBoxOfficeList.do?loadEnd=0&searchType=search&sSearchYearFrom={year}&sMultiMovieYn=N&sRepNationCd={nation_code}"
                driver.get(url)

                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table.tbl_comm tbody tr")))
                time.sleep(2)

                rows = driver.find_elements(By.CSS_SELECTOR, "table.tbl_comm tbody tr")

                count = 0
                for row in rows:
                    if count >= 25: break
                    
                    cols = row.find_elements(By.TAG_NAME, "td")
                    if len(cols) < 8: continue

                    title = row.find_element(By.CSS_SELECTOR, "span.txt_ellip").text.strip()
                    revenue = cols[3].text.replace(",", "").strip()
                    audience = cols[5].text.replace(",", "").strip()
                    screens = cols[7].text.replace(",", "").strip()

                    result_data.append({
                        "범주": category_name,
                        "연도": year,
                        "순위": count + 1,
                        "영화명": title,
                        "매출액": int(revenue) if revenue.isdigit() else 0,
                        "관객수": int(audience) if audience.isdigit() else 0,
                        "스크린수": int(screens) if screens.isdigit() else 0
                    })
                    count += 1

        df = pd.DataFrame(result_data)
        df.to_csv("market_overview.csv", index=False, encoding="utf-8-sig")

    finally:
        driver.quit()
'''
    st.code(excerpt.strip(), language="python")

elif page == "자료 해석":
    st.markdown("## 자료 해석")

    # =========================
    # 시장 비교
    # =========================
    st.markdown("### 시장 비교")

    if not m_change.empty:
        show = m_change.copy()
        show["매출액_전"] = show["매출액_전"].apply(_fmt_money)
        show["매출액_후"] = show["매출액_후"].apply(_fmt_money)
        show["매출액_증감률"] = show["매출액_증감률"].apply(_fmt_pct)
        show["관객수_전"] = show["관객수_전"].apply(_fmt_int)
        show["관객수_후"] = show["관객수_후"].apply(_fmt_int)
        show["관객수_증감률"] = show["관객수_증감률"].apply(_fmt_pct)

        st.dataframe(
            show[["범주","매출액_전","매출액_후","매출액_증감률","관객수_전","관객수_후","관객수_증감률"]]
            .rename(columns={
                "범주":"구분",
                "매출액_전":"매출액 코로나 전",
                "매출액_후":"매출액 코로나 후",
                "매출액_증감률":"매출 변화율",
                "관객수_전":"관객수 코로나 전",
                "관객수_후":"관객수 코로나 후",
                "관객수_증감률":"관객 변화율",
            }),
            use_container_width=True
        )

        melt = []
        for _, r in m_change.iterrows():
            melt += [
                {"구분": r["범주"], "지표":"매출액", "변화율": r["매출액_증감률"]},
                {"구분": r["범주"], "지표":"관객수", "변화율": r["관객수_증감률"]},
            ]
        melt = pd.DataFrame(melt)
        _group_bar_pct(melt, x="지표", y="변화율", color="구분", title="시장 변화율")

        st.info(f"메모: {market_memo(m_change)}")

    st.markdown("### 연도별 추세")
    _line(m_year.sort_values(["범주","연도"]), "연도", "관객수", "범주", "관객수 추세")
    # 연도별 추세 요약: 최근(2025) vs 2019 비교(가능하면)
    try:
        y19 = m_year[m_year["연도"]==2019].set_index("범주")["관객수"]
        y25 = m_year[m_year["연도"]==2025].set_index("범주")["관객수"]
        if ("전체" in y19.index) and ("전체" in y25.index):
            msg = f"전체 관객수는 2019년 {_fmt_int(y19['전체'])}에서 2025년 {_fmt_int(y25['전체'])}로 변화"
            if ("국산" in y19.index) and ("국산" in y25.index):
                msg += f", 국산은 2019년 {_fmt_int(y19['국산'])}에서 2025년 {_fmt_int(y25['국산'])}로 변화"
            st.info(f"메모: {msg}.")
    except:
        pass

    _line(m_year.sort_values(["범주","연도"]), "연도", "매출액", "범주", "매출액 추세")
    try:
        y19s = m_year[m_year["연도"]==2019].set_index("범주")["매출액"]
        y25s = m_year[m_year["연도"]==2025].set_index("범주")["매출액"]
        if ("전체" in y19s.index) and ("전체" in y25s.index):
            msg = f"전체 매출액은 2019년 {_fmt_money(y19s['전체'])}에서 2025년 {_fmt_money(y25s['전체'])}로 변화"
            if ("국산" in y19s.index) and ("국산" in y25s.index):
                msg += f", 국산은 2019년 {_fmt_money(y19s['국산'])}에서 2025년 {_fmt_money(y25s['국산'])}로 변화"
            st.info(f"메모: {msg}.")
    except:
        pass

    st.divider()

    # =========================
    # 장르별 타격 비교
    # =========================
    st.markdown("### 장르별 타격 비교")

    order = []
    if "전체" in g_change["범주"].unique(): order.append("전체")
    if "국산" in g_change["범주"].unique(): order.append("국산")
    for s in sorted(g_change["범주"].unique()):
        if s not in order:
            order.append(s)

    for scope in order:
        tmp = g_change[g_change["범주"]==scope].copy().sort_values("관객수_증감률")
        st.markdown(f"#### {scope}")

        tab = tmp.copy()
        tab["관객수_증감률"] = tab["관객수_증감률"].apply(_fmt_pct)
        tab["매출액_증감률"] = tab["매출액_증감률"].apply(_fmt_pct)
        st.dataframe(
            tab[["장르","관객수_증감률","매출액_증감률"]]
            .rename(columns={"관객수_증감률":"관객 변화율","매출액_증감률":"매출 변화율"}),
            use_container_width=True, height=420
        )

        _thin_barh(tmp, "관객수_증감률", "장르", f"{scope} 장르 관객 변화율")
        st.info(f"메모: {genre_memo(tmp)}")

    st.divider()

    # =========================
    # 온라인
    # =========================
    st.markdown("### 장르별 온라인 타격 비교")

    if not o_change.empty:
        o_sorted = o_change.sort_values("점유율_변화")
        show = o_sorted.copy()
        show["점유율_전"] = show["점유율_전"].map(lambda v: f"{v*100:.2f}%")
        show["점유율_후"] = show["점유율_후"].map(lambda v: f"{v*100:.2f}%")
        show["점유율_변화"] = show["점유율_변화"].map(lambda v: f"{v*100:+.2f}%p")
        st.dataframe(
            show.rename(columns={"점유율_전":"점유율 코로나 전","점유율_후":"점유율 코로나 후","점유율_변화":"점유율 변화"}),
            use_container_width=True, height=420
        )

        tmp = o_sorted.copy()
        _thin_barh(tmp, "점유율_변화", "장르", "온라인 점유율 변화")
        st.info(f"메모: {online_memo(o_change)}")

        if not summary_df.empty and x_thr is not None and y_thr is not None:
            st.markdown("### 개봉 전략 구분")
            st.info(
                "메모: 기준선은 오프라인 관객 변화율 하위 33% 지점과 온라인 점유율 변화 중앙값으로 설정"
            )

            plot_df = summary_df.rename(columns={
                "오프라인 관객 변화율": "offline",
                "온라인 점유율 변화": "online"
            })[["장르","offline","online"]].copy()

            _scatter_with_lines(
                plot_df,
                x="offline", y="online", text="장르",
                title=f"{base_label} 오프라인 관객 변화율과 온라인 점유율 변화",
                vline=x_thr, hline=y_thr
            )
            st.info(f"메모: {strategy_memo(summary_df)}")

            st.markdown("### 정리 표")
            out = summary_df.copy()
            out["오프라인 관객 변화율"] = out["오프라인 관객 변화율"].apply(_fmt_pct)
            out["오프라인 매출 변화율"] = out["오프라인 매출 변화율"].apply(_fmt_pct)
            out["온라인 점유율 변화"] = out["온라인 점유율 변화"].apply(lambda v: "NA" if pd.isna(v) else f"{v*100:+.2f}%p")
            out = out[["장르","오프라인 관객 변화율","오프라인 매출 변화율","온라인 점유율 변화","제언"]].copy()

            order_map = {"OTT":0, "극장":1, "추가 검토":2, "재검토":3}
            out["_o"] = out["제언"].map(order_map).fillna(9)
            out = out.sort_values(["_o","장르"]).drop(columns=["_o"])

            st.dataframe(out, use_container_width=True, height=520)

    else:
        st.warning("online_stats.csv에서 점유율 데이터를 충분히 읽지 못했습니다.")

elif page == "프로젝트 성과":
    st.markdown("## 프로젝트 성과")
    st.markdown("""
본 프로젝트는 코로나 전과 코로나 후 구간을 기준으로 영화 산업의 수요와 규모가 동시에 약화되었는지 여부를 시장 지표로 확인하고, 그 충격이 장르별로 어떻게 분화되는지까지 연결하여 정리하였다. 시장 단계에서는 매출과 관객을 함께 비교함으로써 티켓 가격 상승 효과가 포함된 매출 지표를 단독으로 해석하지 않고, 실제 수요 기반의 변화가 동반되었는지를 검증하였다. 또한 동일한 비교 구간에서 전체와 국산을 분리하여 확인함으로써 국산 시장의 취약성이 상대적으로 더 크게 나타나는지 여부를 점검할 수 있도록 구성하였다.  

장르 단계에서는 동일한 시장 충격이 장르별로 균질하게 나타나지 않는다는 점에 초점을 두고, 수집된 주요 장르를 기준으로 관객 변화율과 매출 변화율을 동시에 제시하였다. 이를 통해 특정 장르는 극장 수요 감소가 집중되는 반면, 일부 장르는 상대적으로 방어되는 양상이 확인될 수 있으며, 장르별로 개봉 전략을 분리해야 하는 근거를 확보하였다.  

마지막으로 온라인 점유율 변화를 결합하여 극장에서 타격이 큰 장르 중 온라인에서의 구조적 변화가 제한적인 장르를 식별하고, 채널 전환 전략을 제언 가능한 형태로 정리하였다. 아래 표는 자료 해석 단계에서 도출된 장르별 지표와 제언 결과를 요약한 것이다.
""")
    if not summary_df.empty:
        out = summary_df.copy()
        out["오프라인 관객 변화율"] = out["오프라인 관객 변화율"].apply(_fmt_pct)
        out["오프라인 매출 변화율"] = out["오프라인 매출 변화율"].apply(_fmt_pct)
        out["온라인 점유율 변화"] = out["온라인 점유율 변화"].apply(lambda v: "NA" if pd.isna(v) else f"{v*100:+.2f}%p")
        out = out[["장르","오프라인 관객 변화율","오프라인 매출 변화율","온라인 점유율 변화","제언"]].copy()
        st.dataframe(out, use_container_width=True, height=520)
    else:
        st.info("온라인 결합 요약표 생성 불가")

elif page == "프로젝트 기대효과":
    st.markdown("## 프로젝트 기대효과")
    st.markdown("""
본 분석은 영화 산업의 변화가 단일 지표로 설명되지 않는다는 점을 전제로, 시장 지표와 장르 지표를 연결하여 전략적 의사결정에 바로 활용 가능한 형태로 정리했다는 점에서 기대효과가 있다. 첫째, 장르별 관객 변화율을 기반으로 극장 개봉 리스크가 높은 장르와 상대적으로 유지되는 장르를 구분할 수 있어, 제작·배급 단계에서 창구 전략을 장르 단위로 설계할 수 있다. 둘째, 온라인 점유율 변화를 결합하면 극장 타격이 큰 장르 중에서도 온라인에서 상대적 선호 구조가 유지되는 장르를 식별할 수 있어, 극장 중심 전략 대신 OTT 공개 전략으로 전환하는 근거를 제공할 수 있다.  

셋째, 전체와 국산을 분리하여 비교한 구조는 국산 시장이 어떤 구간에서 더 취약한지를 장르 단위로 점검하는 데 활용될 수 있다. 이는 국산 영화 산업의 포트폴리오 재구성, 투자 우선순위 조정, 배급 전략 재설계에 실무적으로 연결될 수 있는 형태의 결과물이다. 아래 요약표는 장르별 지표와 제언 결과를 기준으로, 개봉 채널 전략을 정리하는 데 바로 활용할 수 있다.
""")

    if not summary_df.empty:
        out = summary_df.copy()
        out["오프라인 관객 변화율"] = out["오프라인 관객 변화율"].apply(_fmt_pct)
        out["온라인 점유율 변화"] = out["온라인 점유율 변화"].apply(lambda v: "NA" if pd.isna(v) else f"{v*100:+.2f}%p")
        out = out[["장르","오프라인 관객 변화율","온라인 점유율 변화","제언"]].copy()

        order_map = {"OTT":0, "극장":1, "추가 검토":2, "재검토":3}
        out["_o"] = out["제언"].map(order_map).fillna(9)
        out = out.sort_values(["_o","장르"]).drop(columns=["_o"])

        st.dataframe(out, use_container_width=True, height=520)
    else:
        st.info("요약표 생성 불가")


Overwriting movie.py
